# 01 — Speech Tokenization & Forced Alignment (DTW)

## The Core Problem

When a person says **"Hello World"**, the audio might be 1 second long.
At 16,000 Hz that is 16,000 raw samples. After Log-Mel feature extraction: ~100 frames.
The text has 2 words, 10 characters, or roughly 6 phonemes.

**The mismatch:**
```
Audio frames:  [f1][f2][f3]...[f100]       <- 100 frames
Text tokens:   [ H ][ e ][ l ][ l ][ o ]  <- 10 characters
```

We need to know: **which audio frames correspond to which text token?**

This notebook covers the **classical approach**: represent speech as phoneme tokens, then align them to audio using Dynamic Time Warping (DTW).

| Method | This notebook | Next notebook |
|--------|---------------|---------------|
| DTW Forced Alignment | Yes | |
| CTC | | Yes |
| Attention-based (Whisper) | | Yes |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)
print("Notebook ready")

## 1. What Are Tokens in Speech?

Different systems use different token granularities:

```
Text:        "Hello"
             /
Characters:  H  e  l  l  o               <- Tacotron 2, simple ASR
Phonemes:    HH EH L OW                  <- Classical ASR, better pronunciation
BPE tokens:  [Hello]  or  [Hell][o]      <- Whisper (same BPE as GPT-2)
```

### Phonemes: the sound units of language

A **phoneme** is the smallest unit of sound that changes word meaning.
English has ~44 phonemes but only 26 letters — spelling does not equal pronunciation.

| Word | Spelling | Phonemes (ARPAbet) |
|------|---------|-------------------|
| "cat" | c-a-t | K AE T |
| "hello" | h-e-l-l-o | HH EH L OW |
| "through" | t-h-r-o-u-g-h | TH R UW |
| "knight" | k-n-i-g-h-t | N AY T |

This is why TTS/ASR systems often convert text to phonemes first —
"through" and "threw" map to the same phoneme sequence.

In [ ]:
# G2P (Grapheme-to-Phoneme) examples — no install needed
examples = {
    "hello":     ["HH", "EH0", "L", "OW1"],
    "world":     ["W", "ER1", "L", "D"],
    "through":   ["TH", "R", "UW1"],
    "knight":    ["N", "AY1", "T"],
    "speech":    ["S", "P", "IY1", "CH"],
    "alignment": ["AH0", "L", "AY1", "N", "M", "AH0", "N", "T"],
}

print("Grapheme-to-Phoneme (G2P) conversion:")
print("-" * 50)
for word, phones in examples.items():
    print(f"  {word:12s}  ->  {' '.join(phones)}")

print()
print("'through' and 'threw' sound the same -> same phonemes:")
print("  through: TH R UW1")
print("  threw:   TH R UW1")

In [ ]:
# Visualize: Text -> Phoneme -> Audio Frame Mapping
text      = "Hello World"
phonemes  = ["HH", "EH", "L", "OW", "W", "ER", "L", "D"]
durations = [8, 12, 10, 15, 8, 18, 10, 12]

fig, axes = plt.subplots(3, 1, figsize=(14, 6), gridspec_kw={'height_ratios': [1, 1.5, 1]})

# Row 1: Characters
chars = list(text.replace(" ", "_"))
for i, ch in enumerate(chars):
    axes[0].add_patch(plt.Rectangle((i, 0), 0.9, 0.8, color='#AED6F1', ec='gray', lw=0.8))
    axes[0].text(i + 0.45, 0.4, ch, ha='center', va='center', fontsize=11, fontweight='bold')
axes[0].set_xlim(0, len(chars)); axes[0].set_ylim(0, 1)
axes[0].set_title('Level 1: Characters (spelling)', fontsize=11, loc='left')
axes[0].axis('off')

# Row 2: Phonemes (width proportional to duration)
colors = plt.cm.Set3(np.linspace(0, 1, len(phonemes)))
x = 0; total = sum(durations)
for ph, dur, col in zip(phonemes, durations, colors):
    w = dur / total * len(chars)
    axes[1].add_patch(plt.Rectangle((x, 0.1), w - 0.05, 0.8, color=col, ec='gray'))
    axes[1].text(x + w/2, 0.5, ph, ha='center', va='center', fontsize=10, fontweight='bold')
    x += w
axes[1].set_xlim(0, len(chars)); axes[1].set_ylim(0, 1)
axes[1].set_title('Level 2: Phonemes (width = duration in frames)', fontsize=11, loc='left')
axes[1].axis('off')

# Row 3: Audio frames colored by phoneme
n_frames = sum(durations)
frame_colors = []
for ph_idx, dur in enumerate(durations):
    frame_colors.extend([colors[ph_idx]] * dur)
for i in range(n_frames):
    axes[2].add_patch(plt.Rectangle((i * len(chars)/n_frames, 0),
                                     len(chars)/n_frames - 0.01, 0.8,
                                     color=frame_colors[i], ec='gray', lw=0.3))
axes[2].set_xlim(0, len(chars)); axes[2].set_ylim(0, 1)
axes[2].set_title(f'Level 3: Audio frames ({n_frames} frames at 100fps)', fontsize=11, loc='left')
axes[2].axis('off')

plt.suptitle('Text -> Phoneme -> Audio Frame: The Alignment Challenge', fontsize=13)
plt.tight_layout(); plt.show()
print(f"Characters: {len(text.replace(' ',''))}  |  Phonemes: {len(phonemes)}  |  Frames: {n_frames}")

## 2. Forced Alignment — Dynamic Time Warping (DTW)

**Scenario:** You have audio AND the transcript.
You want to find exactly **when** each phoneme starts and ends.

This is **Forced Alignment** — force the known text onto the audio.

### Dynamic Time Warping (DTW)

DTW finds the optimal non-linear mapping between two sequences.
For speech: align acoustic frames to phoneme prototype vectors.

```
Audio:     [A1][A2][A3][A4][A5][A6][A7][A8]
                |       |           |
DTW path:  HH  HH   EH EH EH    L   L   L
                |       |           |
Phonemes:  [ HH ]     [ EH ]      [ L ]
```

Key constraint: **monotonic** — phonemes must appear in order (no skipping back).

In [ ]:
def dtw_align(query, reference):
    T, N = len(query), len(reference)

    # Cost matrix: distance between each frame and each phoneme prototype
    dist = np.zeros((T, N))
    for t in range(T):
        for n in range(N):
            dist[t, n] = np.sum((query[t] - reference[n]) ** 2)

    # Accumulated cost (dynamic programming)
    acc = np.full((T, N), np.inf)
    acc[0, 0] = dist[0, 0]
    for t in range(1, T): acc[t, 0] = acc[t-1, 0] + dist[t, 0]
    for n in range(1, N): acc[0, n] = acc[0, n-1] + dist[0, n]
    for t in range(1, T):
        for n in range(1, N):
            acc[t, n] = dist[t, n] + min(acc[t-1, n],      # stay in phoneme (more frames)
                                          acc[t-1, n-1],    # advance both
                                          acc[t, n-1])      # skip to next phoneme

    # Traceback: find optimal path
    path = [(T-1, N-1)]
    t, n = T-1, N-1
    while t > 0 or n > 0:
        if t == 0:   n -= 1
        elif n == 0: t -= 1
        else:
            step = np.argmin([acc[t-1, n], acc[t-1, n-1], acc[t, n-1]])
            if step == 0:   t -= 1
            elif step == 1: t -= 1; n -= 1
            else:            n -= 1
        path.append((t, n))
    path.reverse()
    return dist, acc, path

# Simulate 40 audio frames, 5 phonemes, 13-dim MFCC features
T, N, D = 40, 5, 13
phoneme_names = ["HH", "EH", "L", "OW", "W"]
boundaries    = [0, 6, 14, 22, 32, 40]

audio_feats = np.zeros((T, D))
for i in range(N):
    proto = np.random.randn(D)
    for t in range(boundaries[i], boundaries[i+1]):
        audio_feats[t] = proto + 0.3 * np.random.randn(D)

phoneme_feats = np.array([audio_feats[boundaries[i]:boundaries[i+1]].mean(0)
                           for i in range(N)])

dist, acc, path = dtw_align(audio_feats, phoneme_feats)
frame_to_phoneme = np.zeros(T, dtype=int)
for t, n in path:
    frame_to_phoneme[t] = n

print("DTW alignment computed.")
print(f"  Total frames: {T}  |  Phonemes: {N}  |  Path length: {len(path)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: accumulated cost matrix with path
im = axes[0].imshow(acc.T, origin='lower', aspect='auto', cmap='Blues')
path_t, path_n = zip(*path)
axes[0].plot(path_t, path_n, 'r-', linewidth=2.5, label='DTW path')
axes[0].scatter(path_t, path_n, color='red', s=20, zorder=5)
axes[0].set_xlabel('Audio frame (t)'); axes[0].set_ylabel('Phoneme (n)')
axes[0].set_yticks(range(N)); axes[0].set_yticklabels(phoneme_names, fontsize=11)
axes[0].set_title('DTW Accumulated Cost + Alignment Path')
axes[0].legend(fontsize=10); fig.colorbar(im, ax=axes[0])

# Right: resulting frame-to-phoneme mapping
colors = plt.cm.Set2(np.linspace(0, 1, N))
for t in range(T):
    axes[1].add_patch(plt.Rectangle((t, 0), 1, 1,
                                     color=colors[frame_to_phoneme[t]], ec='gray', lw=0.2))
for n in range(N):
    frames = [t for t in range(T) if frame_to_phoneme[t] == n]
    if frames:
        axes[1].text((min(frames)+max(frames))/2 + 0.5, 0.5, phoneme_names[n],
                     ha='center', va='center', fontsize=13, fontweight='bold')
        axes[1].axvline(min(frames), color='black', lw=2)

axes[1].set_xlim(0, T); axes[1].set_ylim(0, 1)
axes[1].set_xlabel('Audio frame')
axes[1].set_title('DTW Forced Alignment Result')
axes[1].axis('off')

plt.suptitle('Forced Alignment with DTW — "Hello"', fontsize=13)
plt.tight_layout(); plt.show()

print("Phoneme boundaries:")
for n in range(N):
    frames = [t for t in range(T) if frame_to_phoneme[t] == n]
    print(f"  {phoneme_names[n]:4s}: frames {min(frames):2d}-{max(frames):2d}  "
          f"({(max(frames)-min(frames)+1)*10} ms)")

## 3. Real-World Forced Alignment: MFA

In practice, researchers use **Montreal Forced Aligner (MFA)** — a tool that combines:
- **Acoustic model** (trained HMM-GMM): predicts phoneme likelihoods per frame
- **Pronunciation dictionary**: maps words to phoneme sequences
- **Viterbi alignment**: efficient DP to find the best phoneme boundaries

MFA is used to generate ground-truth duration labels for **FastSpeech 2** training.

```bash
# Install
pip install montreal-forced-aligner

# Run alignment
mfa align /data/audio/ english_us_arpa english_us_arpa /data/aligned/
# Outputs: TextGrid files with precise phoneme start/end timestamps
```

### What a TextGrid looks like

```
"hello world" spoken over 1 second:

HH: 0.00 - 0.08 sec
EH: 0.08 - 0.20 sec
L:  0.20 - 0.30 sec
OW: 0.30 - 0.45 sec
(pause)
W:  0.48 - 0.56 sec
ER: 0.56 - 0.72 sec
L:  0.72 - 0.82 sec
D:  0.82 - 1.00 sec
```

## Summary

| Concept | Key point |
|---------|-----------|
| **Phoneme** | Smallest sound unit; 44 in English (ARPAbet / IPA) |
| **G2P** | Converts spelling to phonemes; handles irregular pronunciations |
| **DTW** | Non-linear alignment via dynamic programming; monotonic constraint |
| **MFA** | Production forced alignment tool; produces phoneme timestamps |

**Limitation of forced alignment:** requires the transcript in advance.
For real-world ASR (unknown transcripts), we need CTC or attention-based methods.

**Next:** 02-CTC-and-Attention — align audio to text **without** knowing the transcript in advance.